# Мордовкин Максим Михайлович, Архипов И.С., N4170, Методы Оптимизации, Лабораторная №1, "Задача ЛП"

## Задача

Рассматривается задача финансирования набора проектов за счёт набора источников и собственных средств. Известны денежные потоки каждого проекта и каждого источника по периодам $t = 0, \ldots, T-1$.

**Переменные:**
- $x_j$ - доля вложения в проект $j$, $0 \le x_j \le 1$;
- $y_i$ - доля использования источника $i$, $0 \le y_i \le 1$.

**Требуется:** выбрать доли так, чтобы максимизировать итоговый накопленный капитал, не уходя в минус по балансу ни в одном периоде.

**Требования к генерируемым данным:**

1. Все числа кратны 5; минимум 5 проектов, 5 источников, 4 периода.
2. Для каждого проекта/источника есть хотя бы один положительный и один отрицательный поток.
3. По каждому проекту: сумма потоков > 0 и доход ≥ 1.10 × вложений.
4. По каждому источнику: сумма потоков < 0 и выплаты ≥ 1.15 × начального поступления.
5. Полузакрытая система: сумма вложений в проекты ≥ сумма средств от источников + собственные средства.

# Генерация данных

In [ ]:
import numpy as np
import random

def generate_data(seed=15, n_projects=5, n_sources=5, n_periods=4):
    """Генерирует данные задачи финансирования проектов.

    Возвращает:
      projects  - матрица (n_projects, n_periods),
                  t=0 - вложение (отриц.), t>=1 - доходы (полож.)
      sources   - матрица (n_sources, n_periods),
                  t=0 - получение (полож.), t>=1 - выплаты (отриц.)
      own_funds - вектор длины n_periods (собственные средства)
    """
    rng = random.Random(seed)
    max_attempts = 1000

    for _ in range(max_attempts):
        # Проекты
        projects = []
        for j in range(n_projects):
            # Вложение на t=0: кратное 5, от -100 до -200
            inv = -5 * rng.randint(20, 40)
            # Доходы на t=1..T-1
            returns = [5 * rng.randint(10, 30) for _ in range(n_periods - 1)]
            # Доход >= 1.10 * |вложение|
            min_total = int(1.10 * abs(inv))
            while sum(returns) < min_total:
                idx = rng.randint(0, n_periods - 2)
                returns[idx] += 5
            projects.append([inv] + returns)

        # Источники
        sources = []
        for i in range(n_sources):
            init = 5 * rng.randint(5, 15)   # 25..75
            payments = [-5 * rng.randint(5, 15) for _ in range(n_periods - 1)]
            # Суммарные выплаты >= 1.15 * начального поступления
            min_paid = int(1.15 * init)
            while sum(-p for p in payments) < min_paid:
                idx = rng.randint(0, n_periods - 2)
                payments[idx] -= 5
            sources.append([init] + payments)

        # Собственные средства
        own_funds = [0] * n_periods
        own_funds[0] = 5 * rng.randint(5, 15)   # 25..75

        # Проверка полузакрытой системы
        total_inv = sum(abs(p[0]) for p in projects)
        total_src = sum(s[0] for s in sources) + own_funds[0]
        if total_inv >= total_src:
            return (np.array(projects, dtype=float),
                    np.array(sources, dtype=float),
                    np.array(own_funds, dtype=float))

    raise RuntimeError("Не удалось сгенерировать данные за "
                       f"{max_attempts} попыток")


projects, sources, own_funds = generate_data(seed=15)

print("Проекты (строки - проекты, столбцы - периоды):")
print(projects)
print("\nИсточники:")
print(sources)
print("\nСобственные средства:", own_funds)

Проекты (строки — проекты, столбцы — периоды):
[[-130.   50.  130.   55.]
 [-125.   85.   50.   55.]
 [-120.  105.   85.   65.]
 [-150.  120.  105.   90.]
 [-160.   90.  105.   85.]]

Источники:
[[ 40. -50. -50. -40.]
 [ 45. -65. -55. -40.]
 [ 70. -60. -55. -60.]
 [ 30. -60. -70. -50.]
 [ 60. -70. -50. -75.]]

Собственные средства: [60.  0.  0.  0.]


# Проверка требований к данным

In [ ]:
def check_data(projects, sources, own_funds):
    """Проверяет выполнение всех требований к данным."""
    n, T = projects.shape
    m = sources.shape[0]

    print("Проверка данных")

    # 1. Кратность 5
    ok5 = all((v % 5 == 0) for v in projects.flatten()) and \
          all((v % 5 == 0) for v in sources.flatten()) and \
          all((v % 5 == 0) for v in own_funds)
    print(f"1. Все числа кратны 5:               {ok5}")

    # 2. У каждого проекта/источника есть + и -
    ok_signs = True
    for j in range(n):
        if not (any(projects[j] > 0) and any(projects[j] < 0)):
            ok_signs = False
    for i in range(m):
        if not (any(sources[i] > 0) and any(sources[i] < 0)):
            ok_signs = False
    print(f"2. Есть + и - потоки:                 {ok_signs}")

    # 3. Проекты: сумма > 0 и доход >= 1.10 * вложений
    ok_proj = True
    for j in range(n):
        total = projects[j].sum()
        inv = abs(projects[j, 0])
        if not (total > 0 and total >= 0.10 * inv):
            ok_proj = False
    print(f"3. Проекты прибыльны (>= 10%):        {ok_proj}")

    # 4. Источники: сумма < 0 и выплаты >= 1.15 * b_i0
    ok_src = True
    for i in range(m):
        total = sources[i].sum()
        paid = -sources[i, 1:].sum()
        init = sources[i, 0]
        if not (total < 0 and paid >= 1.15 * init):
            ok_src = False
    print(f"4. Источники прибыльны (>= 15%):      {ok_src}")

    # 5. Полузакрытая система
    total_inv = sum(abs(p[0]) for p in projects)
    total_src = sum(s[0] for s in sources) + own_funds[0]
    print(f"5. Вложения >= источников + собств.:  {total_inv} >= {total_src}  ({total_inv >= total_src})")


check_data(projects, sources, own_funds)

Проверка данных
1. Все числа кратны 5:               True
2. Есть + и - потоки:                 True
3. Проекты прибыльны (>= 10%):        True
4. Источники прибыльны (>= 15%):      True
5. Вложения >= источников + собств.:  685.0 >= 305.0  (True)


## Обоснование целевой функции

В качестве целевой функции берём итоговый накопленный капитал к концу
горизонта планирования. Так как реинвестирования в модели нет (свободные
деньги просто остаются на счёте), итог равен сумме всех денежных потоков
за все периоды:

Итог = c0 + Σ_j x_j * Σ_t a_jt + Σ_i y_i * Σ_t b_it

Собственные средства c0 — это константа, она не влияет на выбор оптимума,
поэтому при оптимизации её можно не учитывать. Получаем:

max Σ_j A_j * x_j + Σ_i B_i * y_i

где A_j = Σ_t a_jt — суммарный доход от проекта j (если вложиться в него полностью),
B_i = Σ_t b_it — суммарные выплаты по источнику i (при полном использовании, B_i < 0).

Смысл простой: каждый проект приносит прибыль (A_j > 0), каждый источник
требует выплат (|B_i|). Оптимум — это баланс между доходностью проектов и
стоимостью источников, при условии что в каждом периоде не уходим в минус.

## Математическая модель

**Индексы:**
- $j = 1, \ldots, n$ - проекты;
- $i = 1, \ldots, m$ - источники;
- $t = 0, \ldots, T-1$ - периоды.

**Переменные:**
- $x_j \in [0, 1]$ - доля вложения в проект $j$;
- $y_i \in [0, 1]$ - доля использования источника $i$.

**Целевая функция:**

$$\max\ \sum_{j=1}^{n} A_j\, x_j + \sum_{i=1}^{m} B_i\, y_i$$

где $A_j = \sum_t a_{jt}$, $B_i = \sum_t b_{it}$.

**Ограничения:**

$$\sum_{j=1}^{n} a_{jt}\, x_j + \sum_{i=1}^{m} b_{it}\, y_i + c_t \ge 0, \quad t = 0, \ldots, T-1$$

$$0 \le x_j \le 1, \quad j = 1, \ldots, n$$

$$0 \le y_i \le 1, \quad i = 1, \ldots, m$$

**Смысл:** в каждом периоде суммарный денежный поток (от проектов, источников и собственных средств) должен быть неотрицательным - нельзя уйти в минус.

In [ ]:
from scipy.optimize import linprog
import time, tracemalloc

def solve_scipy(projects, sources, own_funds):
    """Решает задачу через scipy.optimize.linprog (метод HiGHS)."""
    n, T = projects.shape
    m = sources.shape[0]

    # Целевая функция: максимизируем A_j x_j + B_i y_i
    # linprog минимизирует, поэтому берём -c
    A_j = projects.sum(axis=1)   # суммарный доход проекта
    B_i = sources.sum(axis=1)    # суммарные выплаты источника (отриц.)
    c = -np.concatenate([A_j, B_i])

    # Ограничения по периодам: A_ub z <= b_ub
    # Для каждого t: -sum(a_jt x_j) - sum(b_it y_i) <= c_t
    A_ub = np.zeros((T, n + m))
    for t in range(T):
        A_ub[t, :n] = -projects[:, t]
        A_ub[t, n:] = -sources[:, t]
    b_ub = own_funds.copy()

    # Границы: 0 <= x_j, y_i <= 1
    bounds = [(0, 1)] * (n + m)

    res = linprog(c=c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
    return res

res_scipy = solve_scipy(projects, sources, own_funds)

x_scipy = res_scipy.x[:projects.shape[0]]
y_scipy = res_scipy.x[projects.shape[0]:]
f_scipy = -res_scipy.fun

print("РЕШЕНИЕ ЧЕРЕЗ SCIPY")
print("success:", res_scipy.success)
print("\nДоли вложения в проекты:")
for j, v in enumerate(x_scipy, 1):
    print(f"  x_{j} = {v:.4f}")
print("\nДоли использования источников:")
for i, v in enumerate(y_scipy, 1):
    print(f"  y_{i} = {v:.4f}")
print(f"\nМаксимум целевой функции: f = {f_scipy:.4f}")

РЕШЕНИЕ ЧЕРЕЗ SCIPY
success: True

Доли вложения в проекты:
  x_1 = 0.0000
  x_2 = 0.0000
  x_3 = 0.5000
  x_4 = 0.0000
  x_5 = 0.0000

Доли использования источников:
  y_1 = 0.0000
  y_2 = 0.0000
  y_3 = 0.0000
  y_4 = 0.0000
  y_5 = 0.0000

Максимум целевой функции: f = 67.5000


In [ ]:
!apt-get install -y coinor-cbc

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  coinor-libcbc3.1 coinor-libcgl1 coinor-libclp1 coinor-libcoinutils3v5
  coinor-libosi1v5
The following NEW packages will be installed:
  coinor-cbc coinor-libcbc3.1 coinor-libcgl1 coinor-libclp1
  coinor-libcoinutils3v5 coinor-libosi1v5
0 upgraded, 6 newly installed, 0 to remove and 2 not upgraded.
Need to get 2,911 kB of archives.
After this operation, 7,857 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 coinor-libcoinutils3v5 amd64 2.11.4+repack1-4 [469 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 coinor-libosi1v5 amd64 0.108.9+ds-1 [329 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/universe amd64 coinor-libclp1 amd64 1.17.9+ds-1 [904 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble/universe amd64 coinor-libcgl1 amd64 0.60.8+ds-1 [423 kB]
Get:5 http://archive

In [ ]:
!pip install "pulp[cbc]" -q
!apt-get install -y coinor-cbc -qq

import pulp


def solve_pulp(projects, sources, own_funds):
    """Решает задачу через PuLP. Совместимо с PuLP 3.x и 4.x."""
    n, T = projects.shape
    m = sources.shape[0]

    prob = pulp.LpProblem("Project_Financing", pulp.LpMaximize)

    # Переменные
    # В PuLP 4.0 LpVariable удалён, используем prob.add_variable.
    # В PuLP 3.x - наоборот. Пробуем оба варианта.
    def make_var(name):
        if hasattr(prob, "add_variable"):
            return prob.add_variable(name, lowBound=0, upBound=1)
        return pulp.LpVariable(name, lowBound=0, upBound=1)

    x = [make_var(f"x_{j+1}") for j in range(n)]
    y = [make_var(f"y_{i+1}") for i in range(m)]

    # Целевая функция
    A_j = projects.sum(axis=1)
    B_i = sources.sum(axis=1)
    prob += (pulp.lpSum(A_j[j] * x[j] for j in range(n)) +
             pulp.lpSum(B_i[i] * y[i] for i in range(m))), "Total_Profit"

    # Ограничения по балансу в каждом периоде
    for t in range(T):
        prob += (pulp.lpSum(projects[j, t] * x[j] for j in range(n)) +
                 pulp.lpSum(sources[i, t] * y[i] for i in range(m)) +
                 own_funds[t] >= 0), f"Balance_t{t}"

    # Решаем
    # Пробуем разные способы вызова солвера в зависимости от версии PuLP.
    solved = False

    # Вариант 1: PULP_CBC_CMD (PuLP 3.x и некоторые сборки 4.x)
    if hasattr(pulp, "PULP_CBC_CMD"):
        try:
            prob.solve(pulp.PULP_CBC_CMD(msg=0))
            solved = True
        except Exception:
            pass

    # Вариант 2: COIN_CMD с явным путём (PuLP 4.x + apt-get install cbc)
    if not solved and hasattr(pulp, "COIN_CMD"):
        try:
            prob.solve(pulp.COIN_CMD(path="/usr/bin/cbc", msg=0))
            solved = True
        except Exception:
            pass

    # Вариант 3: просто solve() - пусть сам ищет
    if not solved:
        prob.solve()

    return prob, x, y


# Запуск
prob, x_pulp, y_pulp = solve_pulp(projects, sources, own_funds)

x_vals = [v.varValue for v in x_pulp]
y_vals = [v.varValue for v in y_pulp]

# Значение целевой функции: пробуем разные способы
try:
    f_pulp = pulp.value(prob.objective)
except Exception:
    f_pulp = sum(projects.sum(axis=1)[j] * x_vals[j] for j in range(len(x_vals))) \
           + sum(sources.sum(axis=1)[i] * y_vals[i] for i in range(len(y_vals)))

# Статус: в PuLP 4.0 у LpProblem нет .status - берём через .sol_status или просто пропускаем
status = getattr(prob, "status", None)
if status is None:
    status_str = "solved"
else:
    status_map = {0: "Not Solved", 1: "Optimal", -1: "Infeasible",
                  -2: "Unbounded", -3: "Undefined"}
    status_str = status_map.get(status, "Unknown")

print("РЕШЕНИЕ ЧЕРЕЗ PULP")
print("status:", status_str)
print("\nДоли вложения в проекты:")
for j, v in enumerate(x_vals, 1):
    print(f"  x_{j} = {v:.4f}")
print("\nДоли использования источников:")
for i, v in enumerate(y_vals, 1):
    print(f"  y_{i} = {v:.4f}")
print(f"\nМаксимум целевой функции: f = {f_pulp:.4f}")

РЕШЕНИЕ ЧЕРЕЗ PULP
status: solved

Доли вложения в проекты:
  x_1 = 0.0000
  x_2 = 0.0000
  x_3 = 0.5000
  x_4 = 0.0000
  x_5 = 0.0000

Доли использования источников:
  y_1 = 0.0000
  y_2 = 0.0000
  y_3 = 0.0000
  y_4 = 0.0000
  y_5 = 0.0000

Максимум целевой функции: f = 67.5000


In [ ]:
def benchmark(func, *args, n_runs=10, **kwargs):
    """Замеряет время и пиковую память при вызове функции."""
    # Прогрев
    func(*args, **kwargs)

    times = []
    peaks = []
    for _ in range(n_runs):
        tracemalloc.start()
        t0 = time.perf_counter()
        func(*args, **kwargs)
        t1 = time.perf_counter()
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        times.append(t1 - t0)
        peaks.append(peak)

    return {
        'time_avg': np.mean(times),
        'time_std': np.std(times),
        'mem_avg':  np.mean(peaks),
        'mem_max':  np.max(peaks),
    }


bench_scipy = benchmark(solve_scipy, projects, sources, own_funds)
bench_pulp  = benchmark(solve_pulp,  projects, sources, own_funds)

print("СРАВНЕНИЕ БИБЛИОТЕК")
print(f"{'Метрика':<28}{'scipy':<15}{'PuLP':<15}")
print(f"{'Среднее время, мс':<28}"
      f"{bench_scipy['time_avg']*1000:<15.3f}"
      f"{bench_pulp['time_avg']*1000:<15.3f}")
print(f"{'Std времени, мс':<28}"
      f"{bench_scipy['time_std']*1000:<15.3f}"
      f"{bench_pulp['time_std']*1000:<15.3f}")
print(f"{'Средняя пик. память, КБ':<28}"
      f"{bench_scipy['mem_avg']/1024:<15.1f}"
      f"{bench_pulp['mem_avg']/1024:<15.1f}")
print(f"{'Макс. пик. память, КБ':<28}"
      f"{bench_scipy['mem_max']/1024:<15.1f}"
      f"{bench_pulp['mem_max']/1024:<15.1f}")

print("\nПроверка совпадения результата:")
print(f"  f_scipy = {f_scipy:.6f}")
print(f"  f_pulp  = {f_pulp:.6f}")
print(f"  разница = {abs(f_scipy - f_pulp):.2e}")

СРАВНЕНИЕ БИБЛИОТЕК
Метрика                     scipy          PuLP           
Среднее время, мс           7.652          22.130         
Std времени, мс             0.213          1.289          
Средняя пик. память, КБ     10.5           20.9           
Макс. пик. память, КБ       10.5           20.9           

Проверка совпадения результата:
  f_scipy = 67.500000
  f_pulp  = 67.500000
  разница = 0.00e+00


## Выводы

### Математическая модель

Построена модель ЛП для задачи оптимального финансирования проектов:

$$\max \sum_j A_j x_j + \sum_i B_i y_i$$

при ограничениях на неотрицательность баланса в каждом периоде и границах $0 \le x_j, y_i \le 1$.

### Результаты

- Получены доли вложения в проекты $x_j^*$ и доли использования источников $y_i^*$.
- Значение целевой функции (итоговый накопленный капитал) совпадает для обоих решателей.

### Сравнение scipy и PuLP

| Критерий | scipy | PuLP |
|---|---|---|
| Скорость | обычно быстрее (компилируемое ядро HiGHS) | медленнее (Python-обёртка над CBC) |
| Память | меньше | больше |
| Удобство моделирования | требует ручной сборки матрицы | декларативный синтаксис |
| Установка | входит в состав scipy | требует `pip install pulp` |

**Итог:** для небольших задач оба инструмента работают за миллисекунды. Для больших моделей `scipy` быстрее за счёт оптимизированного ядра, а `PuLP` удобнее в написании ограничений.